<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB02_dotplots_pairwise_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB02_dotplots_pairwise_alignment.ipynb)

# NB02 — Dot plots and pairwise sequence alignment

## From visual similarity to an optimal alignment

**Biological question:** How do we move from *seeing* possible similarity between two sequences to calculating an optimal alignment, and how do the parameters we choose change the answer?

This exercise consolidates the lecture material on:

- dot plots and sliding windows;
- background noise and window size;
- global versus local pairwise alignment;
- dynamic programming: initialization, matrix fill, and traceback;
- substitution matrices such as BLOSUM62;
- linear versus affine gap penalties.

The biological examples use cytochrome *c* proteins because they provide both close and more divergent comparisons.

> **Before beginning:** In Colab choose **File → Save a copy in Drive** and work in your own copy.

## Learning goals

By the end of this notebook you should be able to:

1. Explain what a dot in a dot plot means.
2. Explain why a diagonal indicates similarity in the same residue order.
3. Predict how increasing a sliding-window size or score threshold changes dot-plot noise.
4. Distinguish global (Needleman–Wunsch) from local (Smith–Waterman) alignment.
5. Explain the three dynamic-programming stages: initialization, matrix fill, and traceback.
6. Explain how a substitution matrix supplies the score for a diagonal move.
7. Predict how gap-opening and gap-extension penalties affect an alignment.
8. Change selected parameters, rerun the analysis, and interpret the resulting figures rather than accepting software defaults blindly.

## 1. Install and import the tools

Biopython provides the pairwise-alignment algorithms and substitution matrices. NumPy, pandas, and Matplotlib are used to calculate and visualize the matrices.

In [ ]:
%pip install -q biopython pandas matplotlib

from pathlib import Path
from urllib.request import urlretrieve
import csv
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from Bio import Align, SeqIO
from Bio.Align import substitution_matrices

print("Tools are ready.")

## 2. Connect Google Drive and locate the course folder

The setup recognizes both the normal student location and the instructor `Teaching` location. The temporary notebook number for this exercise is **NB02**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

COURSE_FOLDER_NAME = "BIOINFO4-5203-F26"
NOTEBOOK_ID = "NB02_dotplots_pairwise_alignment"

candidate_course_dirs = [
    Path("/content/drive/MyDrive") / COURSE_FOLDER_NAME,
    Path("/content/drive/MyDrive/Teaching") / COURSE_FOLDER_NAME,
]

existing = [p for p in candidate_course_dirs if p.exists()]
COURSE_DIR = existing[0] if existing else candidate_course_dirs[0]
COURSE_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = COURSE_DIR / "Data" / NOTEBOOK_ID
OUTPUT_DIR = COURSE_DIR / "Outputs" / NOTEBOOK_ID
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Course folder:", COURSE_DIR)
print("Data folder:  ", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

## 3. Obtain the cytochrome *c* course sequence set

During the temporary renumbering, this notebook first looks in its new NB02 data folder. If the file is not there, it looks for the earlier NB04 copy. If neither is present, it attempts to download the existing course copy from GitHub.

This keeps the exercise usable before the repository is reorganized.

In [ ]:
FASTA_NAME = "cytochrome_c_course_set.fasta"
FASTA_PATH = DATA_DIR / FASTA_NAME

legacy_path = COURSE_DIR / "Data" / "NB04_pairwise_alignment" / FASTA_NAME

legacy_url = (
    "https://raw.githubusercontent.com/RobBurnap/"
    "Bioinformatics-MICR4203-MICR5203/main/"
    "data/NB04_pairwise_alignment/cytochrome_c_course_set.fasta"
)

if FASTA_PATH.exists():
    print("Using NB02 copy:", FASTA_PATH)
elif legacy_path.exists():
    shutil.copy2(legacy_path, FASTA_PATH)
    print("Copied the earlier NB04 data file into the NB02 data folder.")
else:
    print("Course FASTA not found in Drive; trying the current GitHub course copy...")
    try:
        urlretrieve(legacy_url, FASTA_PATH)
        print("Downloaded:", FASTA_PATH)
    except Exception as exc:
        raise FileNotFoundError(
            "Could not locate cytochrome_c_course_set.fasta. "
            "Place that file in Data/NB02_dotplots_pairwise_alignment/ "
            "and rerun this cell."
        ) from exc

records = list(SeqIO.parse(FASTA_PATH, "fasta"))

print(f"\nLoaded {len(records)} sequences:")
for record in records:
    print(f"  {record.id}: {len(record.seq)} aa")

if len(records) < 3:
    raise ValueError("This exercise expects the three-sequence cytochrome c course set.")

# Part I — Dot plots

The lecture begins with a simple rule: compare every position in one sequence with every position in the other. An identical pair produces a dot. A coherent diagonal therefore represents residues occurring in the same order.

A dot plot displays **possible matches**. It does not itself choose the optimal alignment.

## 4. A literal one-residue dot plot

We begin with the short lecture-style sequences so that every dot can be inspected.

`THISSEQUENCE` versus `THISISASEQUENCE`

Look especially for:

- the strong main diagonal;
- isolated off-diagonal dots;
- displaced/parallel diagonal pieces caused by a change in register.

In [ ]:
def exact_match_matrix(seq_x, seq_y):
    return np.array([
        [1 if x == y else 0 for x in seq_x]
        for y in seq_y
    ], dtype=int)

def plot_exact_dotplot(seq_x, seq_y, title, save_path=None):
    matrix = exact_match_matrix(seq_x, seq_y)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(matrix, origin="upper", aspect="equal", vmin=0, vmax=1)

    ax.set_xticks(range(len(seq_x)))
    ax.set_xticklabels(list(seq_x))
    ax.set_yticks(range(len(seq_y)))
    ax.set_yticklabels(list(seq_y))

    ax.set_xlabel("Sequence 1")
    ax.set_ylabel("Sequence 2")
    ax.set_title(title)

    # Put a visible marker only where the letters match.
    y, x = np.where(matrix == 1)
    ax.scatter(x, y, marker="o", facecolors="none", edgecolors="black")

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()
    return matrix

lecture_x = "THISSEQUENCE"
lecture_y = "THISISASEQUENCE"

lecture_dot_matrix = plot_exact_dotplot(
    lecture_x,
    lecture_y,
    "Exact-match dot plot: window size = 1",
    OUTPUT_DIR / "01_exact_match_dotplot.png"
)

### Check your understanding

1. Why does a diagonal mean more than a collection of isolated dots?
2. What does a displaced diagonal imply about residue order and register?
3. Why can repeated letters create background dots that are not part of the best alignment?

## 5. Sliding windows suppress random matches

For a DNA alphabet with equal, independent base frequencies:

- window size 1: random exact-match probability = \( (1/4)^1 = 25\% \)
- window size 3: random exact-match probability = \( (1/4)^3 = 1.5625\% \)

The assumptions are deliberately simple. The important concept is that requiring a longer ordered match suppresses random background.

### Student task — change one parameter

In the next cell, change **only** `WINDOW_SIZE` and run the cell three times:

1. `WINDOW_SIZE = 1`
2. `WINDOW_SIZE = 3`
3. `WINDOW_SIZE = 5`

Leave `dna_1` and `dna_2` unchanged.

Each run saves a separate figure named `02_DNA_window_1.png`, `02_DNA_window_3.png`, or `02_DNA_window_5.png`. Compare the figures and the printed number of exact-window hits.

**Important for interpretation:** the x- and y-axes always show the full nucleotide positions of the original sequences. Increasing the window size changes the number of possible matching windows; it does **not** shorten the plotted sequence axes. Each plotted point is placed at the center of an exact matching window.

For your comparison, be prepared to explain briefly how increasing `WINDOW_SIZE` changes (1) background noise and (2) the number of possible window comparisons.


In [ ]:
# ===== STUDENT PARAMETER: CHANGE ONLY THIS VALUE =====
# Run this cell with WINDOW_SIZE = 1, then 3, then 5.
WINDOW_SIZE = 1

# Keep these sequences unchanged for this exercise.
dna_1 = "ATGCCTAG"
dna_2 = "ATGCCTAG"

def exact_window_hits(seq_x, seq_y, window):
    if window < 1:
        raise ValueError("Window size must be >= 1.")
    if window > min(len(seq_x), len(seq_y)):
        raise ValueError("Window is longer than one of the sequences.")

    hits = np.zeros(
        (len(seq_y) - window + 1, len(seq_x) - window + 1),
        dtype=int
    )

    for y in range(hits.shape[0]):
        for x in range(hits.shape[1]):
            if seq_x[x:x+window] == seq_y[y:y+window]:
                hits[y, x] = 1
    return hits

hits = exact_window_hits(dna_1, dna_2, WINDOW_SIZE)

# Plot matching windows using the coordinate system of the ORIGINAL sequences.
# A hit is shown at the center of its matching window. This keeps the axes
# fixed at the full sequence lengths for every WINDOW_SIZE.
y_start, x_start = np.where(hits == 1)
x_plot = x_start + 1 + (WINDOW_SIZE - 1) / 2
y_plot = y_start + 1 + (WINDOW_SIZE - 1) / 2

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    x_plot,
    y_plot,
    marker="o",
    facecolors="none",
    edgecolors="black"
)

# Keep the axes tied to the actual nucleotide positions, not to the number
# of possible windows.
ax.set_xlim(0.5, len(dna_1) + 0.5)
ax.set_ylim(len(dna_2) + 0.5, 0.5)
ax.set_aspect("equal", adjustable="box")
ax.set_xticks(range(1, len(dna_1) + 1))
ax.set_yticks(range(1, len(dna_2) + 1))

ax.set_title(f"Exact DNA window matches: window = {WINDOW_SIZE}")
ax.set_xlabel("Position in sequence 1")
ax.set_ylabel("Position in sequence 2")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / f"02_DNA_window_{WINDOW_SIZE}.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

p_random = (1/4) ** WINDOW_SIZE
possible_windows = hits.shape[0] * hits.shape[1]

print(f"Window size: {WINDOW_SIZE}")
print(f"Sequence lengths: {len(dna_1)} nt × {len(dna_2)} nt")
print(f"Random exact-window probability under the simple 1/4 model: {100*p_random:.4f}%")
print(f"Possible window comparisons: {possible_windows}")
print(f"Observed exact-window hits: {hits.sum()}")
print("Plot axes remain fixed at the full nucleotide sequence lengths.")


## 6. Protein dot plots: use substitution scores, not just identity

For proteins, conservative substitutions can be biologically meaningful even when the letters are not identical. We therefore score a diagonal window using **BLOSUM62**.

For each candidate diagonal window, the program sums the BLOSUM62 score of the aligned residue pairs. A point is retained only when the sum reaches the threshold.

### Student parameters

The next cell has three parameters at the top:

- `PROTEIN_MATRIX`: leave this as `"BLOSUM62"` unless instructed otherwise.
- `PROTEIN_WINDOW`: controls how many consecutive residues are scored together.
- `PROTEIN_THRESHOLD`: controls how high the window score must be before a point is plotted.

Start with the supplied values. Then change **one parameter at a time** and rerun the cell. A larger window asks similarity to persist over a longer segment; a higher threshold makes the plot more stringent.

For your comparison, record which parameter you changed and describe whether the main diagonal became clearer or whether more background points appeared.


In [ ]:
# ===== STUDENT PARAMETERS =====
# Leave the matrix as BLOSUM62 unless instructed otherwise.
# Change ONE of the two numeric parameters at a time, then rerun this cell.
PROTEIN_MATRIX = "BLOSUM62"
PROTEIN_WINDOW = 5       # larger = similarity must persist across more residues
PROTEIN_THRESHOLD = 10   # larger = fewer, more stringent plotted matches

record_by_id = {record.id: record for record in records}

cyt549_id = next(r.id for r in records if "1E29" in r.id)
human_id = next(r.id for r in records if "P99999" in r.id)
tuna_id = next(r.id for r in records if "3CYT" in r.id)

cyt549 = str(record_by_id[cyt549_id].seq)
human = str(record_by_id[human_id].seq)
tuna = str(record_by_id[tuna_id].seq)

# The historical course sequences can contain a leading nonstandard/initiator
# position. Use the same mature-sequence convention as the earlier notebook.
human_mature = human[1:] if human.startswith("M") else human
tuna_mature = tuna[1:] if tuna.startswith("X") else tuna

sub_matrix = substitution_matrices.load(PROTEIN_MATRIX)

def substitution_window_scores(seq_x, seq_y, matrix, window):
    if window > min(len(seq_x), len(seq_y)):
        raise ValueError("Window is longer than one of the sequences.")

    scores = np.zeros(
        (len(seq_y) - window + 1, len(seq_x) - window + 1),
        dtype=float
    )

    for y in range(scores.shape[0]):
        for x in range(scores.shape[1]):
            score = 0.0
            for k in range(window):
                score += matrix[seq_x[x+k], seq_y[y+k]]
            scores[y, x] = score

    return scores

def plot_scored_protein_dotplot(seq_x, seq_y, label_x, label_y, stem):
    scores = substitution_window_scores(
        seq_x, seq_y, sub_matrix, PROTEIN_WINDOW
    )

    # Continuous score landscape
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(scores, origin="upper", aspect="auto")
    fig.colorbar(image, ax=ax, label=f"{PROTEIN_MATRIX} window score")
    ax.set_xlabel(label_x)
    ax.set_ylabel(label_y)
    ax.set_title(
        f"Protein similarity score landscape\n"
        f"window={PROTEIN_WINDOW}, threshold={PROTEIN_THRESHOLD}"
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"{stem}_score_landscape.png",
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()

    # Thresholded dot plot
    mask = scores >= PROTEIN_THRESHOLD
    y, x = np.where(mask)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(x, y, s=8)
    ax.invert_yaxis()
    ax.set_xlabel(label_x)
    ax.set_ylabel(label_y)
    ax.set_title(
        f"Thresholded protein dot plot\n"
        f"{PROTEIN_MATRIX}, window={PROTEIN_WINDOW}, "
        f"threshold={PROTEIN_THRESHOLD}"
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"{stem}_threshold_dotplot.png",
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()

    print(f"{label_x} vs {label_y}: {mask.sum()} windows passed the threshold")
    return scores, mask

print("First: a close comparison (tuna vs human)")
tuna_human_scores, tuna_human_mask = plot_scored_protein_dotplot(
    tuna_mature, human_mature,
    "Tuna cytochrome c", "Human cytochrome c",
    "03_tuna_human"
)

print("\nSecond: a more divergent comparison (cyanobacterial c549 vs human)")
c549_human_scores, c549_human_mask = plot_scored_protein_dotplot(
    cyt549, human_mature,
    "Synechocystis cytochrome c549", "Human cytochrome c",
    "04_c549_human"
)

### Interpret the protein dot plots

Compare the close and divergent pairs.

- Which comparison produces the cleaner diagonal?
- What happens if you lower the threshold?
- What happens if you increase the window?
- Why is a BLOSUM-scored window more informative for proteins than requiring exact identity at every position?
- A repeated domain would often produce multiple parallel diagonal segments. How would that differ visually from a single conserved region?

# Part II — Dynamic programming

A dot plot shows candidate similarities. Dynamic programming chooses an optimal path through a scoring matrix.

For a **linear gap penalty**, a global-alignment cell can be written as

\[
F(i,j)=\max
\begin{cases}
F(i-1,j-1)+s(x_i,y_j)\\
F(i-1,j)+g\\
F(i,j-1)+g
\end{cases}
\]

The three stages are:

1. **Initialization**
2. **Matrix fill**
3. **Traceback**

The next cell implements this explicitly so that you can see the score matrix rather than treating the aligner as a black box.

In [ ]:
# ===== STUDENT PARAMETERS =====
DP_MATCH = 1
DP_MISMATCH = 0
DP_GAP = -0

DP_SEQ_1 = "THISLINE"
DP_SEQ_2 = "ISALIGNED"

def needleman_wunsch_demo(seq_x, seq_y, match=1, mismatch=0, gap=-1):
    n = len(seq_x)
    m = len(seq_y)

    F = np.zeros((m + 1, n + 1), dtype=int)
    trace = np.empty((m + 1, n + 1), dtype=object)

    for j in range(1, n + 1):
        F[0, j] = F[0, j-1] + gap
        trace[0, j] = "L"

    for i in range(1, m + 1):
        F[i, 0] = F[i-1, 0] + gap
        trace[i, 0] = "U"

    trace[0, 0] = "S"

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = match if seq_x[j-1] == seq_y[i-1] else mismatch

            diag = F[i-1, j-1] + s
            up   = F[i-1, j] + gap
            left = F[i, j-1] + gap

            best = max(diag, up, left)
            F[i, j] = best

            # Deterministic tie-breaking for display only.
            if diag == best:
                trace[i, j] = "D"
            elif up == best:
                trace[i, j] = "U"
            else:
                trace[i, j] = "L"

    # Traceback
    i, j = m, n
    path = [(i, j)]
    ax_aln = []
    ay_aln = []

    while i > 0 or j > 0:
        move = trace[i, j]

        if move == "D":
            ax_aln.append(seq_x[j-1])
            ay_aln.append(seq_y[i-1])
            i -= 1
            j -= 1
        elif move == "U":
            ax_aln.append("-")
            ay_aln.append(seq_y[i-1])
            i -= 1
        else:
            ax_aln.append(seq_x[j-1])
            ay_aln.append("-")
            j -= 1

        path.append((i, j))

    return (
        F,
        trace,
        "".join(reversed(ax_aln)),
        "".join(reversed(ay_aln)),
        list(reversed(path))
    )

F, trace, dp_aln_1, dp_aln_2, dp_path = needleman_wunsch_demo(
    DP_SEQ_1, DP_SEQ_2,
    match=DP_MATCH,
    mismatch=DP_MISMATCH,
    gap=DP_GAP
)

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(F, origin="upper", aspect="equal")
fig.colorbar(image, ax=ax, label="Best cumulative score")

for i in range(F.shape[0]):
    for j in range(F.shape[1]):
        ax.text(j, i, str(F[i, j]), ha="center", va="center")

path_y = [i for i, j in dp_path]
path_x = [j for i, j in dp_path]
ax.plot(path_x, path_y, marker="o")

ax.set_xticks(range(len(DP_SEQ_1) + 1))
ax.set_xticklabels(["–"] + list(DP_SEQ_1))
ax.set_yticks(range(len(DP_SEQ_2) + 1))
ax.set_yticklabels(["–"] + list(DP_SEQ_2))
ax.set_xlabel("Sequence 1")
ax.set_ylabel("Sequence 2")
ax.set_title("Needleman–Wunsch score matrix and one optimal traceback")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "05_dynamic_programming_traceback.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

print("Scoring:")
print(" match   =", DP_MATCH)
print(" mismatch=", DP_MISMATCH)
print(" gap     =", DP_GAP)
print("\nOne optimal alignment:")
print(dp_aln_1)
print(dp_aln_2)
print("Final score:", F[-1, -1])

### Parameter experiment: dynamic programming

Change `DP_GAP` and rerun the previous cell.

- Make the gap penalty **more negative**. What happens to the number/location of gaps?
- Make mismatch scores less favorable. Does the traceback change?
- Why can two different alignments sometimes have the same optimal score?

Notice that the matrix stores the **best cumulative score up to each cell**. Traceback then reconstructs one optimal alignment from those stored choices.

# Part III — Global and local biological alignments

Now use Biopython's optimized alignment algorithms on the cytochrome *c* sequences.

A **global** alignment asks for an optimal end-to-end alignment.

A **local** alignment asks for the strongest matching region and permits poor-scoring flanking regions to be abandoned.

In [ ]:
simple_aligner = Align.PairwiseAligner()
simple_aligner.match_score = 2
simple_aligner.mismatch_score = -1
simple_aligner.gap_score = -2

simple_aligner.mode = "global"
global_simple = simple_aligner.align(cyt549, human_mature)[0]

simple_aligner.mode = "local"
local_simple = simple_aligner.align(cyt549, human_mature)[0]

print("SIMPLE GLOBAL ALIGNMENT")
print("Algorithm:", simple_aligner.algorithm if simple_aligner.mode == "global" else "Needleman-Wunsch")
print("Score:", global_simple.score)
print(global_simple)

print("\nSIMPLE LOCAL ALIGNMENT")
print("Score:", local_simple.score)
print(local_simple)

### Decide biologically, not just numerically

Answer before continuing:

1. Which parts of the proteins disappear from the local alignment?
2. If two proteins are expected to be homologous across essentially their entire lengths, which mode is the more natural starting point?
3. If one conserved domain is embedded in otherwise unrelated sequence, which mode is more appropriate?
4. Why should you **not** compare the raw numerical score of a global alignment directly with the raw score of a local alignment and simply choose the larger one?

## 7. Replace identity-only scoring with BLOSUM62 and affine gaps

In a protein alignment the diagonal move need not mean "same letter or mismatch." BLOSUM62 provides a residue-pair score \(s(x_i,y_j)\).

Positive substitution scores are favored relative to the matrix's background model; negative scores are disfavored.

An **affine gap model** distinguishes:

- **gap opening**: the larger cost of starting a new gap;
- **gap extension**: the smaller cost of extending an existing gap.

This reflects the biological idea that one multi-residue insertion/deletion event can be more plausible than many separate single-residue events.

In [ ]:
# ===== STUDENT PARAMETERS =====
ALIGNMENT_MATRIX = "BLOSUM62"
GAP_OPEN = 10.0
GAP_EXTEND = 0.5

matrix = substitution_matrices.load(ALIGNMENT_MATRIX)

protein_aligner = Align.PairwiseAligner()
protein_aligner.substitution_matrix = matrix
protein_aligner.open_gap_score = -GAP_OPEN
protein_aligner.extend_gap_score = -GAP_EXTEND

protein_aligner.mode = "global"
global_blosum = protein_aligner.align(cyt549, human_mature)[0]

protein_aligner.mode = "local"
local_blosum = protein_aligner.align(cyt549, human_mature)[0]

print("Selected BLOSUM62 examples from the lecture:")
for a, b in [("I", "T"), ("I", "H"), ("T", "H"), ("T", "T")]:
    print(f"  s({a},{b}) = {matrix[a,b]:g}")

print("\nGLOBAL — substitution matrix + affine gaps")
print("Algorithm:", Align.PairwiseAligner().algorithm if False else "affine-gap dynamic programming")
print("Score:", global_blosum.score)
print(global_blosum)

print("\nLOCAL — substitution matrix + affine gaps")
print("Score:", local_blosum.score)
print(local_blosum)

## 8. Gap-opening experiment

The lecture compares alignments made with different gap-opening penalties. We will do that systematically.

Keep the extension penalty fixed and vary **only** the opening penalty.

Prediction before running:

> As the gap-opening penalty becomes larger, I predict that the number of separate gaps will __________ because __________.

In [ ]:
GAP_OPEN_VALUES = [5, 10, 15, 25]
FIXED_EXTENSION = 0.5

def alignment_metrics(alignment, matrix):
    s1 = str(alignment[0])
    s2 = str(alignment[1])

    length = len(s1)
    matches = 0
    similar = 0
    gap_columns = 0
    gap_runs = 0
    in_gap = False

    for a, b in zip(s1, s2):
        is_gap = (a == "-" or b == "-")

        if is_gap:
            gap_columns += 1
            if not in_gap:
                gap_runs += 1
            in_gap = True
            continue

        in_gap = False

        if a == b:
            matches += 1
        if matrix[a, b] > 0:
            similar += 1

    return {
        "alignment_length": length,
        "identity_percent": 100 * matches / length,
        "similarity_percent": 100 * similar / length,
        "gap_percent": 100 * gap_columns / length,
        "gap_runs": gap_runs,
        "score": alignment.score,
    }

experiment_rows = []
experiment_alignments = {}

for gap_open in GAP_OPEN_VALUES:
    a = Align.PairwiseAligner()
    a.substitution_matrix = matrix
    a.open_gap_score = -gap_open
    a.extend_gap_score = -FIXED_EXTENSION
    a.mode = "global"

    aln = a.align(cyt549, human_mature)[0]
    metrics = alignment_metrics(aln, matrix)

    experiment_rows.append({
        "gap_open_penalty": gap_open,
        "gap_extension_penalty": FIXED_EXTENSION,
        **metrics
    })
    experiment_alignments[gap_open] = aln

gap_experiment = pd.DataFrame(experiment_rows)
display(gap_experiment.round(2))

gap_experiment.to_csv(
    OUTPUT_DIR / "gap_opening_experiment.tsv",
    sep="\t",
    index=False
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    gap_experiment["gap_open_penalty"],
    gap_experiment["gap_runs"],
    marker="o"
)
ax.set_xlabel("Gap-opening penalty")
ax.set_ylabel("Number of separate gap runs")
ax.set_title("Effect of gap-opening penalty")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "06_gap_open_vs_gap_runs.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    gap_experiment["gap_open_penalty"],
    gap_experiment["gap_percent"],
    marker="o"
)
ax.set_xlabel("Gap-opening penalty")
ax.set_ylabel("Alignment columns containing a gap (%)")
ax.set_title("Gap content changes with gap-opening penalty")
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "07_gap_open_vs_gap_percent.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

for gap_open in [10, 15, 25]:
    print("=" * 72)
    print(
        f"Gap opening = {gap_open}; "
        f"extension = {FIXED_EXTENSION}"
    )
    print(experiment_alignments[gap_open])

### Interpret the gap experiment

1. Did increasing the opening penalty reduce the number of separate gaps?
2. Did it necessarily eliminate all gap residues?
3. Why are **gap opening** and **gap extension** biologically different parameters?
4. Why does changing a scoring parameter change the *optimal alignment* rather than merely changing the printed score?

## 9. Conserved heme-binding motif

Cytochrome *c* proteins provide an immediate functional landmark: the covalent heme-binding motif `CXXCH`.

Find the motif in each course sequence and compare its position with the regions emphasized by the alignment and dot plots.

In [ ]:
motif_rows = []

for record in records:
    motif = re.search(r"C..CH", str(record.seq))
    if motif:
        motif_rows.append({
            "sequence_id": record.id,
            "motif": motif.group(),
            "start_1_based": motif.start() + 1,
            "end_1_based": motif.end()
        })

motif_table = pd.DataFrame(motif_rows)
display(motif_table)

motif_table.to_csv(
    OUTPUT_DIR / "CXXCH_motif_positions.tsv",
    sep="\t",
    index=False
)

# Final synthesis

The major conceptual progression in this notebook is:

**dot plot → scored window → dynamic-programming matrix → traceback → biological alignment**

Complete these questions in your notebook before the exam.

1. What information does a dot plot show that an alignment does not?
2. Why does increasing window size usually reduce background noise?
3. What is the fundamental difference between global and local alignment?
4. In dynamic programming, what do diagonal, up, and left moves represent?
5. Where does BLOSUM62 enter the recurrence?
6. What is the difference between a linear gap penalty and an affine gap penalty?
7. What happened when you increased the gap-opening penalty?
8. Give one example in which changing a parameter changed your biological interpretation.
9. BLAST is a heuristic local similarity-search method. Why is that conceptually different from exhaustive Smith–Waterman dynamic programming?

## 10. Save a reproducibility record and confirm outputs

In [ ]:
parameters = pd.DataFrame([
    ["notebook_id", NOTEBOOK_ID],
    ["input_fasta", FASTA_PATH.name],
    ["DNA_window_last_run", WINDOW_SIZE],
    ["protein_dot_matrix", PROTEIN_MATRIX],
    ["protein_dot_window", PROTEIN_WINDOW],
    ["protein_dot_threshold", PROTEIN_THRESHOLD],
    ["DP_match", DP_MATCH],
    ["DP_mismatch", DP_MISMATCH],
    ["DP_gap", DP_GAP],
    ["alignment_matrix", ALIGNMENT_MATRIX],
    ["gap_open", GAP_OPEN],
    ["gap_extend", GAP_EXTEND],
], columns=["parameter", "value"])

parameters.to_csv(
    OUTPUT_DIR / "run_parameters.tsv",
    sep="\t",
    index=False
)

(OUTPUT_DIR / "global_BLOSUM62_alignment.txt").write_text(
    str(global_blosum) + f"\nScore: {global_blosum.score}\n"
)
(OUTPUT_DIR / "local_BLOSUM62_alignment.txt").write_text(
    str(local_blosum) + f"\nScore: {local_blosum.score}\n"
)

print("Files created in", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(" ✓", p.name)